In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Citirea datelor
data = pd.read_csv("../data/FinalDatasets/dataset_3stations_interpolated.csv", parse_dates=['start', 'end'])


# Extragem ziua și ora
data["day_of_week"] = data["start"].dt.dayofweek
data["hour"] = data["start"].dt.hour
data["month"] = data["start"].dt.month
data["day_of_week"] = data["start"].dt.dayofweek



In [4]:
data

,start,end,pm10,pm2_5,no2,temperature,humidity,wind_speed,pressure,longitude,latitude,location,day_of_week,hour,month
0,2022-01-02 00:00:00,2022-01-02 01:00:00,46.48,42.360000,30.20,4.8,100.0,7.2,1021.3,26.036694,44.447275,Crangasi,6,0,1
1,2022-01-02 00:00:00,2022-01-02 01:00:00,43.33,32.427825,44.18,4.8,100.0,7.2,1021.3,26.127289,44.444925,Piata Obor,6,0,1
2,2022-01-02 00:00:00,2022-01-02 01:00:00,49.69,37.187598,35.06,4.8,100.0,7.2,1021.3,26.098297,44.435044,Universitate,6,0,1
3,2022-01-02 01:00:00,2022-01-02 02:00:00,33.50,30.730000,18.84,3.6,100.0,3.6,1021.8,26.036694,44.447275,Crangasi,6,1,1
4,2022-01-02 01:00:00,2022-01-02 02:00:00,10.62,7.947923,31.10,3.6,100.0,3.6,1021.8,26.127289,44.444925,Piata Obor,6,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19852,2022-12-21 04:00:00,2022-12-21 05:00:00,24.68,18.597614,18.29,-3.3,99.0,10.8,1027.5,26.127289,44.444925,Piata Obor,2,4,12
19853,2022-12-21 04:00:00,2022-12-21 05:00:00,35.19,26.517425,17.65,-3.3,99.0,10.8,1027.5,26.098297,44.435044,Universitate,2,4,12
19854,2022-12-21 05:00:00,2022-12-21 06:00:00,21.42,17.410000,16.81,-3.1,98.0,7.2,1027.1,26.036694,44.447275,Crangasi,2,5,12
19855,2022-12-21 05:00:00,2022-12-21 06:00:00,23.14,17.437147,21.16,-3.1,98.0,7.2,1027.1,26.127289,44.444925,Piata Obor,2,5,12


In [4]:
import numpy as np
import pandas as pd

# Coloane de poluanți
pollution_features = ["pm10", "pm2_5", "no2"]

# Funcția de IDW
def idw_interpolation(lat, lon, hour, day_of_week, month, df, k=2, power=2):
    filtered = df[
        (df["hour"] == hour) &
        (df["day_of_week"] == day_of_week) &
        (df["month"] == month)
    ].copy()

    coords = filtered[["latitude", "longitude"]].values
    target = np.array([lat, lon])
    filtered["distance"] = np.linalg.norm(coords - target, axis=1)

    filtered = filtered[filtered["distance"] > 0]
    nearest = filtered.nsmallest(k, "distance")

    predictions = {}
    for pol in pollution_features:
        weights = 1 / (nearest["distance"] ** power)
        interpolated = np.sum(weights * nearest[pol]) / np.sum(weights)
        predictions[pol] = interpolated

    return predictions

# Copie dataframe pentru completare
df_with_preds = data.copy()

# Liste pentru rezultate
idw_results = []
residuals = []

for i, row in df_with_preds.iterrows():
    loc = row["location"]
    hour = row["hour"]
    day = row["day_of_week"]
    month = row["month"]
    lat = row["latitude"]
    lon = row["longitude"]

    # Alegem locațiile vecine în funcție de loc
    if loc == "Crangasi":
        neighbors = data[(data["location"].isin(["Piata Obor", "Universitate"]))]
    elif loc == "Piata Obor":
        neighbors = data[(data["location"].isin(["Crangasi", "Universitate"]))]
    else:
        idw_results.append({f"idw_{pol}": np.nan for pol in pollution_features})
        residuals.append({f"resid_{pol}": np.nan for pol in pollution_features})
        continue

    # Interpolare
    pred = idw_interpolation(lat, lon, hour, day, month, neighbors, k=2)

    # Reziduuri
    resid = {f"resid_{pol}": row[pol] - pred[pol] for pol in pollution_features}
    pred_renamed = {f"idw_{pol}": pred[pol] for pol in pollution_features}

    idw_results.append(pred_renamed)
    residuals.append(resid)

# Adăugăm coloanele noi
df_with_preds = pd.concat(
    [df_with_preds, pd.DataFrame(idw_results), pd.DataFrame(residuals)], axis=1
)

# Rezultatul este în df_with_preds
# import ace_tools as tools; tools.display_dataframe_to_user(name="Interpolated Data with Residuals", dataframe=df_with_preds)


In [5]:
df_with_preds

,start,end,pm10,pm2_5,no2,temperature,humidity,wind_speed,pressure,longitude,...,location,day_of_week,hour,month,idw_pm10,idw_pm2_5,idw_no2,resid_pm10,resid_pm2_5,resid_no2
0,2022-01-02 00:00:00,2022-01-02 01:00:00,46.48,42.360000,30.20,4.8,100.0,7.2,1021.3,26.036694,...,Crangasi,6,0,1,30.31,22.683762,30.880,16.17,19.676238,-0.680
1,2022-01-02 00:00:00,2022-01-02 01:00:00,43.33,32.427825,44.18,4.8,100.0,7.2,1021.3,26.127289,...,Piata Obor,6,0,1,30.31,22.683762,30.880,13.02,9.744064,13.300
2,2022-01-02 00:00:00,2022-01-02 01:00:00,49.69,37.187598,35.06,4.8,100.0,7.2,1021.3,26.098297,...,Universitate,6,0,1,NaN,NaN,NaN,NaN,NaN,NaN
3,2022-01-02 01:00:00,2022-01-02 02:00:00,33.50,30.730000,18.84,3.6,100.0,3.6,1021.8,26.036694,...,Crangasi,6,1,1,12.34,9.235157,22.335,21.16,21.494843,-3.495
4,2022-01-02 01:00:00,2022-01-02 02:00:00,10.62,7.947923,31.10,3.6,100.0,3.6,1021.8,26.127289,...,Piata Obor,6,1,1,12.34,9.235157,22.335,-1.72,-1.287234,8.765
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19852,2022-12-21 04:00:00,2022-12-21 05:00:00,24.68,18.597614,18.29,-3.3,99.0,10.8,1027.5,26.127289,...,Piata Obor,2,4,12,40.87,30.797589,34.425,-16.19,-12.199975,-16.135
19853,2022-12-21 04:00:00,2022-12-21 05:00:00,35.19,26.517425,17.65,-3.3,99.0,10.8,1027.5,26.098297,...,Universitate,2,4,12,NaN,NaN,NaN,NaN,NaN,NaN
19854,2022-12-21 05:00:00,2022-12-21 06:00:00,21.42,17.410000,16.81,-3.1,98.0,7.2,1027.1,26.036694,...,Crangasi,2,5,12,30.59,23.051094,34.555,-9.17,-5.641094,-17.745
19855,2022-12-21 05:00:00,2022-12-21 06:00:00,23.14,17.437147,21.16,-3.1,98.0,7.2,1027.1,26.127289,...,Piata Obor,2,5,12,30.59,23.051094,34.555,-7.45,-5.613947,-13.395


In [6]:
df = df_with_preds[df_with_preds['location'] != 'Universitate']
df

,start,end,pm10,pm2_5,no2,temperature,humidity,wind_speed,pressure,longitude,...,location,day_of_week,hour,month,idw_pm10,idw_pm2_5,idw_no2,resid_pm10,resid_pm2_5,resid_no2
0,2022-01-02 00:00:00,2022-01-02 01:00:00,46.48,42.360000,30.20,4.8,100.0,7.2,1021.3,26.036694,...,Crangasi,6,0,1,30.310,22.683762,30.880,16.170,19.676238,-0.680
1,2022-01-02 00:00:00,2022-01-02 01:00:00,43.33,32.427825,44.18,4.8,100.0,7.2,1021.3,26.127289,...,Piata Obor,6,0,1,30.310,22.683762,30.880,13.020,9.744064,13.300
3,2022-01-02 01:00:00,2022-01-02 02:00:00,33.50,30.730000,18.84,3.6,100.0,3.6,1021.8,26.036694,...,Crangasi,6,1,1,12.340,9.235157,22.335,21.160,21.494843,-3.495
4,2022-01-02 01:00:00,2022-01-02 02:00:00,10.62,7.947923,31.10,3.6,100.0,3.6,1021.8,26.127289,...,Piata Obor,6,1,1,12.340,9.235157,22.335,-1.720,-1.287234,8.765
6,2022-01-02 02:00:00,2022-01-02 03:00:00,19.34,17.390000,16.96,2.9,100.0,3.6,1021.8,26.036694,...,Crangasi,6,2,1,11.105,8.310893,21.085,8.235,9.079107,-4.125
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19849,2022-12-21 03:00:00,2022-12-21 04:00:00,24.06,18.130413,18.04,-3.3,99.0,7.2,1028.2,26.127289,...,Piata Obor,2,3,12,32.590,24.558194,37.885,-8.530,-6.427781,-19.845
19851,2022-12-21 04:00:00,2022-12-21 05:00:00,21.75,17.500000,9.74,-3.3,99.0,10.8,1027.5,26.036694,...,Crangasi,2,4,12,40.870,30.797588,34.425,-19.120,-13.297588,-24.685
19852,2022-12-21 04:00:00,2022-12-21 05:00:00,24.68,18.597614,18.29,-3.3,99.0,10.8,1027.5,26.127289,...,Piata Obor,2,4,12,40.870,30.797589,34.425,-16.190,-12.199975,-16.135
19854,2022-12-21 05:00:00,2022-12-21 06:00:00,21.42,17.410000,16.81,-3.1,98.0,7.2,1027.1,26.036694,...,Crangasi,2,5,12,30.590,23.051094,34.555,-9.170,-5.641094,-17.745


In [7]:
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# 🔹 1. Pregătim datele – excludem Universitate
# df_train = df[df['location'] != 'Universitate']

df_train = df
# 🔹 2. Definim toate coloanele de input relevante
feature_cols = [
    'idw_pm10', 'idw_pm2_5', 'idw_no2',              # predicțiile brute IDW
    'temperature', 'humidity', 'wind_speed', 'pressure',  # meteo
    'day_of_week', 'hour', 'month',                      # timp
    'latitude', 'longitude'                              # locație
]

# 🔹 3. Setăm X și y
X = df_train[feature_cols]
y = df_train[['resid_pm10', 'resid_pm2_5', 'resid_no2']]

# 🔹 4. Împărțim în train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 🔹 5. Definim modelul
model = MultiOutputRegressor(
    XGBRegressor(
        n_estimators=200,
        max_depth=7,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=1.0,
        random_state=42
    )
)

# 🔹 6. Antrenare
model.fit(X_train, y_train)

# 🔹 7. Predicții
y_pred = model.predict(X_test)

# 🔹 8. Evaluare
mae = mean_absolute_error(y_test, y_pred, multioutput='raw_values')
rmse = np.sqrt(mean_squared_error(y_test, y_pred, multioutput='raw_values'))
r2 = r2_score(y_test, y_pred, multioutput='raw_values')

# 🔹 9. Afișare metrice
for i, pol in enumerate(["PM10", "PM2.5", "NO2"]):
    print(f"\n📊 {pol} - Metrics")
    print(f"MAE:  {mae[i]:.2f}")
    print(f"RMSE: {rmse[i]:.2f}")
    print(f"R²:   {r2[i]:.3f}")



📊 PM10 - Metrics
MAE:  5.35
RMSE: 8.25
R²:   0.712

📊 PM2.5 - Metrics
MAE:  3.41
RMSE: 5.15
R²:   0.667

📊 NO2 - Metrics
MAE:  7.05
RMSE: 10.15
R²:   0.713


In [8]:
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor

# 🔹 1. Subset de parametri pentru Grid Search
param_grid = {
    'estimator__n_estimators': [100, 200],
    'estimator__max_depth': [5, 7],
    'estimator__learning_rate': [0.05, 0.1],
    'estimator__subsample': [0.8, 1.0],
    'estimator__colsample_bytree': [0.8, 1.0]
}

# 🔹 2. Wrapper pentru XGBoost în MultiOutputRegressor
base_model = MultiOutputRegressor(
    XGBRegressor(objective='reg:squarederror', random_state=42)
)

# 🔹 3. Grid Search
grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    scoring='r2',
    cv=3,
    verbose=2,
    n_jobs=-1
)

# 🔹 4. Pornim căutarea
grid_search.fit(X_train, y_train)

# 🔹 5. Afișăm cei mai buni parametri
print("\n✅ Best parameters found:")
print(grid_search.best_params_)

# 🔹 6. Evaluăm modelul final
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred, multioutput='raw_values')
rmse = np.sqrt(mean_squared_error(y_test, y_pred, multioutput='raw_values'))
r2 = r2_score(y_test, y_pred, multioutput='raw_values')

for i, pol in enumerate(["PM10", "PM2.5", "NO2"]):
    print(f"\n📊 {pol} - Metrics (after GridSearch)")
    print(f"MAE:  {mae[i]:.2f}")
    print(f"RMSE: {rmse[i]:.2f}")
    print(f"R²:   {r2[i]:.3f}")


Fitting 3 folds for each of 32 candidates, totalling 96 fits

✅ Best parameters found:
{'estimator__colsample_bytree': 1.0, 'estimator__learning_rate': 0.1, 'estimator__max_depth': 7, 'estimator__n_estimators': 200, 'estimator__subsample': 0.8}

📊 PM10 - Metrics (after GridSearch)
MAE:  5.35
RMSE: 8.25
R²:   0.712

📊 PM2.5 - Metrics (after GridSearch)
MAE:  3.41
RMSE: 5.15
R²:   0.667

📊 NO2 - Metrics (after GridSearch)
MAE:  7.05
RMSE: 10.15
R²:   0.713


In [14]:
import joblib
joblib.dump(best_model, "residual_correction_model.pkl")


['residual_correction_model.pkl']

In [5]:
df_uni = data[data["location"] == "Universitate"].copy()
df_uni

,start,end,pm10,pm2_5,no2,temperature,humidity,wind_speed,pressure,longitude,latitude,location,day_of_week,hour,month
2,2022-01-02 00:00:00,2022-01-02 01:00:00,49.69,37.187598,35.06,4.8,100.0,7.2,1021.3,26.098297,44.435044,Universitate,6,0,1
5,2022-01-02 01:00:00,2022-01-02 02:00:00,13.41,10.035937,22.63,3.6,100.0,3.6,1021.8,26.098297,44.435044,Universitate,6,1,1
8,2022-01-02 02:00:00,2022-01-02 03:00:00,11.00,8.232312,21.19,2.9,100.0,3.6,1021.8,26.098297,44.435044,Universitate,6,2,1
11,2022-01-02 03:00:00,2022-01-02 04:00:00,12.17,9.107931,19.33,2.7,100.0,3.6,1022.1,26.098297,44.435044,Universitate,6,3,1
14,2022-01-02 04:00:00,2022-01-02 05:00:00,11.46,8.576572,24.70,2.2,100.0,3.6,1022.7,26.098297,44.435044,Universitate,6,4,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19844,2022-12-21 01:00:00,2022-12-21 02:00:00,38.43,28.958927,16.22,-3.4,99.0,7.2,1029.7,26.098297,44.435044,Universitate,2,1,12
19847,2022-12-21 02:00:00,2022-12-21 03:00:00,34.46,25.967333,15.73,-3.4,99.0,7.2,1028.9,26.098297,44.435044,Universitate,2,2,12
19850,2022-12-21 03:00:00,2022-12-21 04:00:00,33.99,25.613165,15.48,-3.3,99.0,7.2,1028.2,26.098297,44.435044,Universitate,2,3,12
19853,2022-12-21 04:00:00,2022-12-21 05:00:00,35.19,26.517425,17.65,-3.3,99.0,10.8,1027.5,26.098297,44.435044,Universitate,2,4,12


In [6]:
import numpy as np
import pandas as pd

# Coloanele de poluanți
pollution_features = ["pm10", "pm2_5", "no2"]

# Funcția de IDW
def idw_interpolation(lat, lon, hour, day_of_week, month, df, k=2, power=2):
    filtered = df[
        (df["hour"] == hour) &
        (df["day_of_week"] == day_of_week) &
        (df["month"] == month)
    ].copy()

    coords = filtered[["latitude", "longitude"]].values
    target = np.array([lat, lon])
    filtered["distance"] = np.linalg.norm(coords - target, axis=1)

    filtered = filtered[filtered["distance"] > 0]
    nearest = filtered.nsmallest(k, "distance")

    predictions = {}
    for pol in pollution_features:
        weights = 1 / (nearest["distance"] ** power)
        interpolated = np.sum(weights * nearest[pol]) / np.sum(weights)
        predictions[pol] = interpolated

    return predictions

# Filtrăm doar datele pentru Universitate
df_universitate = data[data["location"] == "Universitate"].copy()

# Pregătim lista cu rezultate IDW
idw_predictions = []

# Trecem prin fiecare rând din Universitate
for _, row in df_universitate.iterrows():
    hour = row["hour"]
    day = row["day_of_week"]
    month = row["month"]
    lat = row["latitude"]
    lon = row["longitude"]

    # Vecini: doar Crangasi și Piata Obor
    neighbors = data[
        (data["location"].isin(["Crangasi", "Piata Obor"]))
    ]

    pred = idw_interpolation(lat, lon, hour, day, month, neighbors, k=2)

    pred_renamed = {f"idw_{pol}": pred[pol] for pol in pollution_features}
    idw_predictions.append(pred_renamed)

# Adăugăm valorile IDW în df_universitate
df_universitate = pd.concat([df_universitate.reset_index(drop=True),
                             pd.DataFrame(idw_predictions)], axis=1)


In [7]:
df_universitate

,start,end,pm10,pm2_5,no2,temperature,humidity,wind_speed,pressure,longitude,latitude,location,day_of_week,hour,month,idw_pm10,idw_pm2_5,idw_no2
0,2022-01-02 00:00:00,2022-01-02 01:00:00,49.69,37.187598,35.06,4.8,100.0,7.2,1021.3,26.098297,44.435044,Universitate,6,0,1,25.635,19.185029,31.195
1,2022-01-02 01:00:00,2022-01-02 02:00:00,13.41,10.035937,22.63,3.6,100.0,3.6,1021.8,26.098297,44.435044,Universitate,6,1,1,8.990,6.728044,23.365
2,2022-01-02 02:00:00,2022-01-02 03:00:00,11.00,8.232312,21.19,2.9,100.0,3.6,1021.8,26.098297,44.435044,Universitate,6,2,1,7.615,5.699005,21.815
3,2022-01-02 03:00:00,2022-01-02 04:00:00,12.17,9.107931,19.33,2.7,100.0,3.6,1022.1,26.098297,44.435044,Universitate,6,3,1,6.815,5.100291,23.465
4,2022-01-02 04:00:00,2022-01-02 05:00:00,11.46,8.576572,24.70,2.2,100.0,3.6,1022.7,26.098297,44.435044,Universitate,6,4,1,7.290,5.455778,24.270
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6614,2022-12-21 01:00:00,2022-12-21 02:00:00,38.43,28.958927,16.22,-3.4,99.0,7.2,1029.7,26.098297,44.435044,Universitate,2,1,12,49.295,37.146247,39.415
6615,2022-12-21 02:00:00,2022-12-21 03:00:00,34.46,25.967333,15.73,-3.4,99.0,7.2,1028.9,26.098297,44.435044,Universitate,2,2,12,37.880,28.544474,41.995
6616,2022-12-21 03:00:00,2022-12-21 04:00:00,33.99,25.613165,15.48,-3.3,99.0,7.2,1028.2,26.098297,44.435044,Universitate,2,3,12,35.645,26.860290,37.850
6617,2022-12-21 04:00:00,2022-12-21 05:00:00,35.19,26.517425,17.65,-3.3,99.0,10.8,1027.5,26.098297,44.435044,Universitate,2,4,12,37.535,28.284499,31.265


In [13]:
data

,start,end,pm10,pm2_5,no2,temperature,humidity,wind_speed,pressure,longitude,latitude,location,day_of_week,hour,month
0,2022-01-02 00:00:00,2022-01-02 01:00:00,46.48,42.360000,30.20,4.8,100.0,7.2,1021.3,26.036694,44.447275,Crangasi,6,0,1
1,2022-01-02 00:00:00,2022-01-02 01:00:00,43.33,32.427825,44.18,4.8,100.0,7.2,1021.3,26.127289,44.444925,Piata Obor,6,0,1
2,2022-01-02 00:00:00,2022-01-02 01:00:00,49.69,37.187598,35.06,4.8,100.0,7.2,1021.3,26.098297,44.435044,Universitate,6,0,1
3,2022-01-02 01:00:00,2022-01-02 02:00:00,33.50,30.730000,18.84,3.6,100.0,3.6,1021.8,26.036694,44.447275,Crangasi,6,1,1
4,2022-01-02 01:00:00,2022-01-02 02:00:00,10.62,7.947923,31.10,3.6,100.0,3.6,1021.8,26.127289,44.444925,Piata Obor,6,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19852,2022-12-21 04:00:00,2022-12-21 05:00:00,24.68,18.597614,18.29,-3.3,99.0,10.8,1027.5,26.127289,44.444925,Piata Obor,2,4,12
19853,2022-12-21 04:00:00,2022-12-21 05:00:00,35.19,26.517425,17.65,-3.3,99.0,10.8,1027.5,26.098297,44.435044,Universitate,2,4,12
19854,2022-12-21 05:00:00,2022-12-21 06:00:00,21.42,17.410000,16.81,-3.1,98.0,7.2,1027.1,26.036694,44.447275,Crangasi,2,5,12
19855,2022-12-21 05:00:00,2022-12-21 06:00:00,23.14,17.437147,21.16,-3.1,98.0,7.2,1027.1,26.127289,44.444925,Piata Obor,2,5,12


In [12]:
import joblib
loaded_model = joblib.load("residual_correction_model.pkl")

features = ['idw_pm10', 'idw_pm2_5', 'idw_no2', 'temperature', 'humidity', 'wind_speed', 'pressure', 'day_of_week', 'hour', 'month', 'latitude', 'longitude']

X_universitate = df_universitate[features]


In [13]:
residual_preds = loaded_model.predict(X_universitate)


In [14]:
df_preds_uni = pd.DataFrame(
    residual_preds,
    columns=["pred_resid_pm10", "pred_resid_pm2_5", "pred_resid_no2"]
)
df_preds_uni

,pred_resid_pm10,pred_resid_pm2_5,pred_resid_no2
0,-0.065592,1.708043,-0.461835
1,1.794980,4.251192,5.004012
2,1.194115,3.963791,6.476766
3,0.219683,1.375491,3.828306
4,0.270984,1.833298,0.680516
...,...,...,...
6614,-16.240688,-9.300734,-15.507804
6615,-12.366433,-10.848261,-16.005928
6616,-9.564402,-6.722017,-15.170117
6617,-10.377947,-7.415342,-13.801105


In [21]:
df_universitate = df_universitate.reset_index(drop=True)
df_universitate = pd.concat([df_universitate, df_preds_uni], axis=1)


In [22]:
df_universitate["pred_pm10"] = df_universitate["idw_pm10"] + df_universitate["pred_resid_pm10"]
df_universitate["pred_pm2_5"] = df_universitate["idw_pm2_5"] + df_universitate["pred_resid_pm2_5"]
df_universitate["pred_no2"] = df_universitate["idw_no2"] + df_universitate["pred_resid_no2"]


In [1]:
df_universitate[80:90]

NameError: name 'df_universitate' is not defined

In [27]:
df_universitate.to_csv("df_universitate_preds.csv", index=False)
